### home work of lesson 26 by Vasyl Storchak

In [1]:
import numpy as np
import h5py

from sklearn.metrics import classification_report, confusion_matrix
from keras.layers import Input, Dense, Activation, ZeroPadding2D, BatchNormalization, Flatten, Conv2D
from keras.layers import AveragePooling2D, MaxPooling2D, Dropout, GlobalMaxPooling2D, GlobalAveragePooling2D
from keras.models import Model

2025-08-18 13:42:01.540314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755513721.557649   12110 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755513721.562640   12110 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755513721.574771   12110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755513721.574787   12110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755513721.574789   12110 computation_placer.cc:177] computation placer alr

In [2]:
def load_dataset(train_path, test_path):
    train_dataset = h5py.File(train_path, "r")
    train_set_x_orig = np.array(train_dataset["train_set_x"][:]) # your train set features
    train_set_y_orig = np.array(train_dataset["train_set_y"][:]) # your train set labels

    test_dataset = h5py.File(test_path, "r")
    test_set_x_orig = np.array(test_dataset["test_set_x"][:]) # your test set features
    test_set_y_orig = np.array(test_dataset["test_set_y"][:]) # your test set labels

    classes = np.array(test_dataset["list_classes"][:]) # the list of classes

    train_set_y_orig = train_set_y_orig.reshape((1, train_set_y_orig.shape[0]))
    test_set_y_orig = test_set_y_orig.reshape((1, test_set_y_orig.shape[0]))

    return train_set_x_orig, train_set_y_orig, test_set_x_orig, test_set_y_orig, classes

In [3]:
train_path = 'data/train_happy.h5'
test_path = 'data/test_happy.h5'
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset(train_path, test_path)

# Normalize image vectors
X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

# Reshape
Y_train = Y_train_orig.T
Y_test = Y_test_orig.T

print("number of training examples = " + str(X_train.shape[0]))
print("number of test examples = " + str(X_test.shape[0]))
print("X_train shape: " + str(X_train.shape))
print("Y_train shape: " + str(Y_train.shape))
print("X_test shape: " + str(X_test.shape))
print("Y_test shape: " + str(Y_test.shape))

number of training examples = 600
number of test examples = 150
X_train shape: (600, 64, 64, 3)
Y_train shape: (600, 1)
X_test shape: (150, 64, 64, 3)
Y_test shape: (150, 1)


In [5]:
def HappyModel(input_shape, activation="relu", dropout_rate=0.5):
    X_input = Input(input_shape)

    X = ZeroPadding2D((3, 3))(X_input)
    X = Conv2D(32, (7, 7), strides=(1, 1), name='conv0')(X)
    X = BatchNormalization(axis=3, name='bn0')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool0')(X)

    X = Conv2D(64, (3, 3), strides=(1, 1), name='conv1')(X)
    X = BatchNormalization(axis=3, name='bn1')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool1')(X)

    X = Conv2D(128, (3, 3), strides=(1, 1), name='conv2')(X)
    X = BatchNormalization(axis=3, name='bn2')(X)
    X = Activation(activation)(X)
    X = MaxPooling2D((2, 2), name='max_pool2')(X)

    X = Flatten()(X)
    X = Dense(128, activation=activation, name='fc1')(X)
    X = Dropout(dropout_rate)(X)

    X = Dense(1, activation='sigmoid', name='fc_out')(X)

    model = Model(inputs=X_input, outputs=X, name='SignClassifier')

    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    
    return model

In [8]:
model = HappyModel(input_shape=(64,64,3), 
                   activation="relu", 
                   dropout_rate=0.5)

history = model.fit(X_train, 
                    Y_train,
                    validation_data=(X_test, Y_test),
                    epochs=200,
                    batch_size=100,
                    verbose=1)

loss, acc = model.evaluate(X_test, 
                           Y_test, 
                           verbose=0)
print(f"Accuracy: {acc:.4f}")

Epoch 1/200


2025-08-18 13:45:25.082083: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 29491200 exceeds 10% of free system memory.
2025-08-18 13:45:25.097188: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 29491200 exceeds 10% of free system memory.
2025-08-18 13:45:27.719064: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 245.69MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-18 13:45:27.719109: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 422.27MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-08-18 13:45:27.758804: I external/local_xla/xla/service/gpu

4/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5163 - loss: 2.7300

I0000 00:00:1755513931.270887   15159 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2025-08-18 13:45:32.424412: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:549] Omitted potentially buggy algorithm eng14{k25=0} for conv %cudnn-conv-bias-activation.10 = (f32[50,64,30,30]{3,2,1,0}, u8[0]{0}) custom-call(f32[50,32,32,32]{3,2,1,0} %bitcast.590, f32[64,32,3,3]{3,2,1,0} %bitcast.597, f32[64]{0} %bitcast.599), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="SignClassifier_1/conv1_1/convolution" source_file="/home/forever/.local/lib/python3.12/site-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_e

6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 326ms/step - accuracy: 0.5408 - loss: 2.4588 - val_accuracy: 0.5467 - val_loss: 0.6812
Epoch 2/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7028 - loss: 0.5754 - val_accuracy: 0.5467 - val_loss: 0.6786
Epoch 3/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.7654 - loss: 0.4570 - val_accuracy: 0.5467 - val_loss: 0.6885
Epoch 4/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.8713 - loss: 0.3311 - val_accuracy: 0.5333 - val_loss: 0.6911
Epoch 5/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8956 - loss: 0.2782 - val_accuracy: 0.5267 - val_loss: 0.7200
Epoch 6/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9154 - loss: 0.2254 - val_accuracy: 0.5067 - val_loss: 0.7055
Epoch 7/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9417 - loss: 0.1614 - val_accuracy: 0.5000 - val_loss: 0.7158
Epoch 8/200
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9375 - loss: 0.1571 - val_accuracy: 0.5200 - val_loss: 0.6925
Epo

2025-08-18 13:46:11.774607: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:549] Omitted potentially buggy algorithm eng14{k25=0} for conv %cudnn-conv-bias-activation.10 = (f32[32,64,30,30]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,32,32,32]{3,2,1,0} %bitcast.590, f32[64,32,3,3]{3,2,1,0} %bitcast.597, f32[64]{0} %bitcast.599), window={size=3x3}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="SignClassifier_1/conv1_1/convolution" source_file="/home/forever/.local/lib/python3.12/site-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false}
2025-08-18 13:46:11.798712: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:549] Omitted potentially buggy

Accuracy: 0.9733


In [9]:
model.save("happy.keras")

In [27]:
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9467


In [ ]:
# bad in 20 epoch(upper for 200, for 20 is 0.44), i want make more epoch

In [28]:
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"Accuracy with 200 epoch: {acc:.4f}")

Accuracy with 200 epoch: 0.9467


In [ ]:
# much better

In [43]:
y_pred_probs = model.predict(X_test)
# y_pred = np.argmax(y_pred_probs, axis=1)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [48]:
y_pred_binary = np.where(y_pred_probs > 0.5, 1, 0)

In [49]:
print("\nClassification Report:")
print(classification_report(Y_test, y_pred_binary, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94        66
           1       0.93      0.98      0.95        84

    accuracy                           0.95       150
   macro avg       0.95      0.94      0.95       150
weighted avg       0.95      0.95      0.95       150



In [50]:
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, y_pred_binary))


Confusion Matrix:
[[60  6]
 [ 2 82]]


In [ ]:
# GOOOOOD test acc.

In [40]:
np.set_printoptions(suppress=True)